In [1]:
import networkx as nx
import pandas as pd
import geopandas as gpd
import osmnx as ox

from genesis.core import BestPointsBase, MetricBase, StateBase
from genesis.metrics import ArrivalTime
from genesis.states import FirstArrivalUnitState
from genesis.tools import list_dict_concat

In [2]:
# Загрузка графа и полигона границ
G = ox.load_graphml("tests/data/test_rng.ml")
area = gpd.read_file("tests/data/test_polygon.gpkg")

In [13]:



class BestNodesGA(BestPointsBase):
    '''
    Поиск лучших узлов для размещения n объектов.

    Расчет производится генетическим алгоритмом.

    ## Область применения
    Определение размещения множества узлов в графе, при котором 
    обеспечиваются наилучшие некоторые целевые метрики. 
    
    ? Дает достаточно точное решение. Хорошо подходит для больших графов.
    '''

    def __init__(self,
                 state_function: StateBase,
                 metric_function: MetricBase,
                 population_size=25,
                 epochs:int=50,
                 appr_val_in_area=0,
                 **kwargs) -> None:
        '''
        ## Аргументы
        `state_function`: StateBase
            функция расчета состояния окружения
        `metric_function`: MetricBase
            Функция расчета метрики
        `best_point_function`: BestPointsBase
            Функция расчета лучшего размещения узла
        `iterations`:int=5
            Количество итераций расчета
        `appr_val_in_area`: int=0
            Доля узлов графа в пределах area, покрытие которой считается приемлемой для принятия расчетной метрики. 
            Если при расчете метрик, из стартового узла (узлов) достижимо меньшее количество узлов,
            то такой узел не рассматривается.
            При расчет размещения нескольких узлов неминуемо возникает ситуация при которой
            часть территории area будет недостижима. Поэтому по умолчанию считается
        '''
        
        self.population_size = population_size
        self.epochs = epochs
        self.appr_val_in_area = appr_val_in_area
        super().__init__(state_function, metric_function, **kwargs)

    def _fit_function(self, env, nodes, area=None, **kwargs):
        '''
        Расчет функции приспособленности
        '''
        # 2.1. Расчет состояния
        times, _ = self.state_function(env=env, points=nodes, **kwargs)

        # 2.2. Расчет стартовой метрики состояния и определение стартового размещения
        best_metric = self.metric_function(times, area=area, **kwargs)

        return best_metric


    def __call__(self,
                 env:nx.MultiDiGraph,
                 dynamic_nodes: list|dict,
                 static_nodes: list|dict = None,
                 before_iters_start_function: callable =None,
                 iter_calc_end_function: callable =None,
                 area: pd.Series = None,
                 **kwargs):
        
        # 0. Проверка корректности пришедших данных
        if not isinstance(env, nx.MultiDiGraph):
            raise TypeError("Тип аргумента `env` должен быть MultiDiGraph!")
        if not static_nodes is None:
            if not ((isinstance(dynamic_nodes,list) and isinstance(static_nodes,list)) or \
                (isinstance(dynamic_nodes,dict) and isinstance(static_nodes,dict))):
                raise TypeError(f'Аргументы `dynamic_nodes` и `static_nodes` должны быть одинакового типа: list|dict'
                                f'Имеют: {type(dynamic_nodes)}, {type(static_nodes)}')
        if not area is None and not isinstance(area, pd.Series):
            raise TypeError(f'Аргумент `area` должен иметь тип `pd.Series`! Имеет {type(area)}')

        # Если передана область которую следует учитывать в расчете
        # установить для нее приемлемый охват равный `self.appr_val_in_area`
        # (0 по умолчанию)
        if not area is None:
            self.best_point_function.__setattr__('appr_val',
                                                 self.appr_val_in_area)

        # 1.1. Если статические узлы не указаны - заменяем значение переменной с None
        # на [] или {} в зависимости от типа данных `dynamic_nodes`
        if static_nodes is None:
            if isinstance(dynamic_nodes,list):
                static_nodes = []
            elif isinstance(dynamic_nodes,dict):
                static_nodes = {}

        # # 1.2. Получение суммарного списка (или словаря) узлов для расчета состояния и метрик
        # start_nodes = list_dict_concat(dynamic_nodes, static_nodes)

        # ===================================== Генетический алгоритм ==============
        # 1. Создание стартовой популяции
        population = [(dynamic_nodes, static_nodes) for _ in range(self.population_size)]
        
        # 2. На каждой эпохе
        for epoch in range(self.epochs):
            # Оценка приспособленности всех особей
            bot_fit = [self._fit_function(env=env, nodes=list_dict_concat(dn, sn)) for dn, sn in population]
            print(bot_fit)

            # Скрещивание


            # Мутация

        return None

nodes = list(G.nodes())
existed_units = {nodes[100]:'A', 
                nodes[2000]:'B', 
                }
new_units = {nodes[3000]:'c',
             nodes[4000]:'d'}


BNGA = BestNodesGA(FirstArrivalUnitState(), ArrivalTime(), population_size=5, epochs=2)


BNGA(env=G, static_nodes=existed_units, dynamic_nodes=new_units)



[7.262001771503841, 7.262001771503841, 7.262001771503841, 7.262001771503841, 7.262001771503841]
[7.262001771503841, 7.262001771503841, 7.262001771503841, 7.262001771503841, 7.262001771503841]
